# RAG para Consulta del Sector Energético Colombiano

## Procesamiento de Lenguaje Natural — Taller Final

Este proyecto implementa un sistema de Retrieval-Augmented Generation (RAG) para responder preguntas sobre documentos del sector energético colombiano, utilizando embeddings, FAISS y un modelo generativo.

## 1. Definición del problema

### ¿Qué hace el pipeline?
Se construye un sistema RAG que permite hacer preguntas en lenguaje natural sobre documentos del sector energético y recuperar respuestas con evidencia textual.

### ¿Por qué es útil?
Los documentos técnicos suelen ser extensos y difíciles de consultar. Este sistema permite acceder rápidamente a información relevante.

### Tipo de tarea NLP
RAG (Retrieval-Augmented Generation)

### Datos utilizados
Se utilizan documentos reales del sector energético colombiano en formato PDF.

### Métrica principal
Recall@k para evaluar si los chunks relevantes son recuperados.

### ¿Por qué RAG?
Porque permite combinar búsqueda semántica con generación de lenguaje natural, mejorando la precisión frente a modelos generativos puros.

In [1]:
# Instalacion de dependencias — ejecutar solo en la primera ejecucion o en Google Colab
# Alternativa: !pip install -r ../requirements.txt

!pip install pypdf sentence-transformers faiss-cpu transformers accelerate pandas matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import gc
import pickle
import random
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # backend no-interactivo para ejecucion sin display
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

C:\Users\sebas\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Semillas para reproducibilidad — resultados identicos entre ejecuciones
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Semillas establecidas: SEED={SEED}")
print(f"GPU disponible: {torch.cuda.is_available()}")

Semillas establecidas: SEED=42
GPU disponible: True


In [4]:
# Deteccion automatica de GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo de computo: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  VRAM: {vram:.1f} GB")
else:
    print("  GPU no disponible, usando CPU")

Dispositivo de computo: CUDA
  GPU : NVIDIA GeForce GTX 1650 Ti
  VRAM: 4.0 GB


## 2. Explicación de datos y preprocesamiento

El corpus está compuesto por documentos del sector energético colombiano en formato PDF. Estos documentos contienen información técnica, económica y estratégica sobre el sector minero-energético, la transición energética, la demanda de energía y el papel del sector en la reactivación económica.

Durante el preprocesamiento se realizaron los siguientes pasos:

1. Carga de los archivos PDF.
2. Extracción del texto página por página.
3. Limpieza básica del texto, eliminando espacios repetidos y saltos de línea innecesarios.
4. Segmentación del corpus en al menos cinco documentos lógicos, dividiendo los PDFs por bloques de páginas.
5. Cálculo de estadísticas descriptivas del corpus, como número de documentos, páginas y longitud promedio de texto.

Esta segmentación permite cumplir con el requisito de construir un corpus con mínimo cinco documentos, manteniendo la trazabilidad hacia el documento original y la página de origen.

CARGA DE PDFs

In [5]:
# Rutas hacia los PDFs usando pathlib
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOCS_DIR = BASE_DIR / "Documents"

pdf_paths = [
    DOCS_DIR / "PEN_2050_UPME.pdf",
    DOCS_DIR / "Sector_Minero_Energetico.pdf",
]

def extract_text_from_pdf(path):
    try:
        reader = PdfReader(path)
        pages = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                pages.append({
                    "documento": Path(path).name,
                    "pagina": i + 1,
                    "texto": text,
                })
        return pages
    except FileNotFoundError:
        print(f"[ERROR] Archivo no encontrado: {path}")
        return []
    except Exception as e:
        print(f"[ERROR] No se pudo leer {Path(path).name}: {e}")
        return []

all_pages = []
for path in pdf_paths:
    pages = extract_text_from_pdf(path)
    all_pages.extend(pages)
    print(f"Cargado: {Path(path).name} — {len(pages)} paginas")

df_pages = pd.DataFrame(all_pages)
df_pages.head()

Cargado: PEN_2050_UPME.pdf — 86 paginas


Cargado: Sector_Minero_Energetico.pdf — 31 paginas


,documento,pagina,texto
0,PEN_2050_UPME.pdf,1,\n \n \nPEN 2050 \nwww.upme.gov.co / Diciembr...
1,PEN_2050_UPME.pdf,2,\n \n \nPEN 2050 \nwww.upme.gov.co / Diciembr...
2,PEN_2050_UPME.pdf,3,\n \n \nPEN 2050 \nwww.upme.gov.co / Diciembr...
3,PEN_2050_UPME.pdf,4,\n \n \nPEN 2050 \nwww.upme.gov.co / Diciembr...
4,PEN_2050_UPME.pdf,5,\n \n \nPEN 2050 \nwww.upme.gov.co / Diciembr...


LIMPIEZA

In [6]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_pages["texto_limpio"] = df_pages["texto"].apply(clean_text)

CREAR “5 DOCUMENTOS”  , se separa aqui en 5 documentos ya que el pdf numero 1 es demasiado grande y tiene mucha info

In [7]:
# Segmentacion mejorada: relativa a cada documento original.
# El indice de pagina se calcula dentro del documento, evitando mezclar PDFs.
PAGINAS_POR_SEGMENTO = 15  # Ajustable segun la densidad del corpus

paginas_por_doc = (
    df_pages.groupby("documento")["pagina"]
    .apply(sorted)
    .to_dict()
)

def asignar_segmento(doc, pagina):
    paginas = paginas_por_doc[doc]
    idx_local = paginas.index(pagina)
    return f"{doc}_parte_{idx_local // PAGINAS_POR_SEGMENTO}"

df_pages["documento_segmentado"] = df_pages.apply(
    lambda row: asignar_segmento(row["documento"], row["pagina"]),
    axis=1,
)

print(f"Segmentos generados: {df_pages['documento_segmentado'].nunique()}")
print(df_pages.groupby("documento_segmentado").size().to_string())

Segmentos generados: 9
documento_segmentado
PEN_2050_UPME.pdf_parte_0               15
PEN_2050_UPME.pdf_parte_1               15
PEN_2050_UPME.pdf_parte_2               15
PEN_2050_UPME.pdf_parte_3               15
PEN_2050_UPME.pdf_parte_4               15
PEN_2050_UPME.pdf_parte_5               11
Sector_Minero_Energetico.pdf_parte_0    15
Sector_Minero_Energetico.pdf_parte_1    15
Sector_Minero_Energetico.pdf_parte_2     1


ANÁLISIS EXPLORATORIO

In [8]:
df_pages["longitud_palabras"] = df_pages["texto_limpio"].str.split().str.len()

print("Cantidad documentos:", df_pages["documento_segmentado"].nunique())
print("Cantidad páginas:", len(df_pages))
print("Promedio palabras:", df_pages["longitud_palabras"].mean())

Cantidad documentos: 9
Cantidad páginas: 117
Promedio palabras: 237.72649572649573


Se genera un histograma

In [9]:
df_pages["longitud_palabras"].hist(bins=30)

plt.title("Distribución de longitud de páginas")
plt.xlabel("Palabras")
plt.ylabel("Frecuencia")
plt.show()

C:\Users\sebas\AppData\Local\Temp\ipykernel_8456\2493079166.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Chunking del corpus

Para construir el sistema RAG, los documentos se dividen en fragmentos más pequeños llamados chunks.

El objetivo es que cada chunk conserve suficiente contexto para responder preguntas, pero sin ser demasiado largo para el modelo de embeddings.

In [10]:
def chunk_text(text, chunk_size=200, overlap=50):
    # Chunking con respeto de limites de oracion.
    # Divide el texto en oraciones y las agrupa hasta alcanzar chunk_size palabras.
    # El overlap retiene las ultimas palabras del chunk anterior para preservar contexto.
    oraciones = re.split(r'(?<=[.!?])\s+', text.strip())
    oraciones = [o.strip() for o in oraciones if o.strip()]

    chunks = []
    buffer = []

    for oracion in oraciones:
        palabras = oracion.split()
        if len(buffer) + len(palabras) <= chunk_size:
            buffer.extend(palabras)
        else:
            if buffer:
                chunks.append(" ".join(buffer))
            if len(palabras) > chunk_size:
                for i in range(0, len(palabras), chunk_size - overlap):
                    sub = palabras[i:i + chunk_size]
                    if sub:
                        chunks.append(" ".join(sub))
                buffer = palabras[-overlap:] if len(palabras) > overlap else palabras[:]
            else:
                buffer = buffer[-overlap:] + palabras

    if buffer:
        chunks.append(" ".join(buffer))

    return chunks

In [11]:
chunked_data = []

for _, row in df_pages.iterrows():
    chunks = chunk_text(row["texto_limpio"])
    
    for i, chunk in enumerate(chunks):
        chunked_data.append({
            "documento": row["documento_segmentado"],
            "documento_original": row["documento"],
            "pagina": row["pagina"],
            "chunk_id": i,
            "texto_chunk": chunk
        })

df_chunks = pd.DataFrame(chunked_data)

df_chunks["longitud_palabras"] = df_chunks["texto_chunk"].str.split().str.len()

df_chunks.head()

,documento,documento_original,pagina,chunk_id,texto_chunk,longitud_palabras
0,PEN_2050_UPME.pdf_parte_0,PEN_2050_UPME.pdf,1,0,PEN 2050 www.upme.gov.co / Diciembre de 2019 0...,23
1,PEN_2050_UPME.pdf_parte_0,PEN_2050_UPME.pdf,2,0,PEN 2050 www.upme.gov.co / Diciembre de 2019 1...,165
2,PEN_2050_UPME.pdf_parte_0,PEN_2050_UPME.pdf,3,0,PEN 2050 www.upme.gov.co / Diciembre de 2019 2...,129
3,PEN_2050_UPME.pdf_parte_0,PEN_2050_UPME.pdf,4,0,PEN 2050 www.upme.gov.co / Diciembre de 2019 1...,200
4,PEN_2050_UPME.pdf_parte_0,PEN_2050_UPME.pdf,4,1,implementación del PEN 2020 - 2050 ..............,149


In [12]:
df_chunks = df_chunks[df_chunks["longitud_palabras"] >= 40].reset_index(drop=True)

print("Cantidad de chunks:", len(df_chunks))
print("Promedio palabras por chunk:", round(df_chunks["longitud_palabras"].mean(), 2))
print("Cantidad de documentos segmentados:", df_chunks["documento"].nunique())

Cantidad de chunks: 225
Promedio palabras por chunk: 147.66
Cantidad de documentos segmentados: 8


In [13]:
estadisticas_corpus = pd.DataFrame({
    "Métrica": [
        "Documentos segmentados",
        "Documentos originales",
        "Páginas procesadas",
        "Chunks generados",
        "Promedio palabras por chunk"
    ],
    "Valor": [
        df_chunks["documento"].nunique(),
        df_chunks["documento_original"].nunique(),
        df_pages.shape[0],
        df_chunks.shape[0],
        round(df_chunks["longitud_palabras"].mean(), 2)
    ]
})

estadisticas_corpus

,Métrica,Valor
0,Documentos segmentados,8.00
1,Documentos originales,2.00
2,Páginas procesadas,117.00
3,Chunks generados,225.00
4,Promedio palabras por chunk,147.66


## 4. Modelo de embeddings

Se utiliza el modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

Este modelo permite crear representaciones vectoriales de textos en varios idiomas, incluyendo español. Se eligió porque el corpus está en español y se requiere búsqueda semántica entre preguntas y fragmentos de documentos.

In [14]:
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(model_name, device=DEVICE)
print(f"Modelo de embeddings cargado en: {DEVICE.upper()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6859.24it/s]


BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo de embeddings cargado en: CUDA


In [15]:
# Se embebe unicamente el texto del chunk, sin metadatos de navegacion.
# Los metadatos (documento, pagina) se conservan en df_chunks para citacion.
texts = df_chunks["texto_chunk"].tolist()

print(f"Textos preparados para embedding: {len(texts)}")
print(f"Ejemplo de texto embebido:\n{texts[0][:200]}...")

Textos preparados para embedding: 225
Ejemplo de texto embebido:
PEN 2050 www.upme.gov.co / Diciembre de 2019 1 La Unidad de Planeación Minero Energética (UPME) es una Unidad Administrativa Especial del orden Nacional, de carácter técnico, adscrita al Ministerio de...


In [16]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Forma de embeddings:", embeddings.shape)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:  12%|█▎        | 1/8 [00:00<00:02,  3.39it/s]

Batches:  25%|██▌       | 2/8 [00:00<00:01,  5.00it/s]

Batches:  38%|███▊      | 3/8 [00:00<00:00,  5.95it/s]

Batches:  50%|█████     | 4/8 [00:00<00:00,  6.51it/s]

Batches:  62%|██████▎   | 5/8 [00:00<00:00,  6.87it/s]

Batches:  75%|███████▌  | 6/8 [00:00<00:00,  6.72it/s]

Batches:  88%|████████▊ | 7/8 [00:01<00:00,  7.04it/s]

Batches: 100%|██████████| 8/8 [00:01<00:00,  7.14it/s]

Forma de embeddings: (225, 384)


## 5. Índice FAISS

Se construye un índice FAISS para realizar búsqueda semántica eficiente.

Los embeddings se normalizan y se usa producto interno como medida de similitud, equivalente a similitud coseno cuando los vectores están normalizados.

In [17]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Chunks indexados:", index.ntotal)

Chunks indexados: 225


### Persistencia del indice FAISS

Se guarda el indice y los metadatos en disco para evitar recomputacion en ejecuciones futuras (~3-5 min ahorrados).

In [18]:
# Guardar indice FAISS y metadatos de chunks
INDEX_PATH = BASE_DIR / "faiss_index.bin"
CHUNKS_PATH = BASE_DIR / "chunks_metadata.pkl"

faiss.write_index(index, str(INDEX_PATH))
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(df_chunks.to_dict("records"), f)

print(f"Indice guardado en: {INDEX_PATH}")
print(f"Metadatos guardados en: {CHUNKS_PATH}")

# Para cargar en ejecuciones futuras (reemplaza la celda de construccion del indice):
# index = faiss.read_index(str(INDEX_PATH))
# with open(CHUNKS_PATH, 'rb') as f:
#     chunks_data = pickle.load(f)
# df_chunks = pd.DataFrame(chunks_data)

Indice guardado en: D:\sebas\Maestria\03 Semestre\01. Apps_ML\NLP_Taller_RAG\faiss_index.bin
Metadatos guardados en: D:\sebas\Maestria\03 Semestre\01. Apps_ML\NLP_Taller_RAG\chunks_metadata.pkl


In [19]:
def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        row = df_chunks.iloc[idx]
        
        results.append({
            "score": float(score),
            "documento": row["documento"],
            "documento_original": row["documento_original"],
            "pagina": row["pagina"],
            "chunk_id": row["chunk_id"],
            "texto_chunk": row["texto_chunk"]
        })
    
    return results

In [20]:
pregunta_demo = "¿Qué importancia tiene el sector minero energético para la economía colombiana?"

resultados = retrieve_chunks(pregunta_demo, top_k=3)

for r in resultados:
    print("Documento:", r["documento"])
    print("Documento original:", r["documento_original"])
    print("Página:", r["pagina"])
    print("Score:", round(r["score"], 3))
    print("Texto:", r["texto_chunk"][:700])
    print("-" * 80)

Documento: Sector_Minero_Energetico.pdf_parte_0
Documento original: Sector_Minero_Energetico.pdf
Página: 4
Score: 0.842
Texto: Proviene de los sectores de Minería, Hidrocarburos y Energía Eléctrica de la IED de las exportaciones Minero - energéticos representaron El sector Minero Energético es clave para la reactivación económica de colombia 7% 34% 56% del pib nacional En 2019 los Sectores Proviene del sector de los servicios públicos 5,7% Entre 2018 -2019 la IED en el sector minero energético creció 21% de los ingresos de la nación 12% @DiegoMesaP
--------------------------------------------------------------------------------
Documento: Sector_Minero_Energetico.pdf_parte_1
Documento original: Sector_Minero_Energetico.pdf
Página: 27
Score: 0.789
Texto: La minería genera empleo para muchos colombianos empleos La actividad minera se desarrolla en municipios alejados de los centros urbanos, generando empleos y desarrollo. 350.000 Guajira 38% PIB Carbón térmico Cesar 39% PIB Carbón térmic

## 6. Preguntas de prueba para evaluar el retrieval

Para evaluar el sistema RAG se construyó un conjunto de preguntas de prueba.  
Cada pregunta tiene palabras clave esperadas que deberían aparecer en los chunks recuperados.

Esta evaluación se enfoca en el retrieval, es decir, en verificar si el sistema recupera fragmentos relevantes del corpus.

In [21]:
preguntas_prueba = [
    {
        "pregunta": "¿Qué importancia tiene el sector minero energético para la economía colombiana?",
        "keywords_esperadas": ["7%", "PIB", "34%", "IED", "56%", "exportaciones"]
    },
    {
        "pregunta": "¿Qué porcentaje de las exportaciones representaron los sectores minero energéticos en 2019?",
        "keywords_esperadas": ["56%", "exportaciones"]
    },
    {
        "pregunta": "¿Cuáles son los principales sectores de consumo de energía en Colombia?",
        "keywords_esperadas": ["transporte", "industrial", "residencial"]
    },
    {
        "pregunta": "¿Qué objetivo tiene el Plan Energético Nacional 2020-2050?",
        "keywords_esperadas": ["modelo energético", "sostenible", "2050"]
    },
    {
        "pregunta": "¿Qué fuentes de energía aumentan su participación en los escenarios energéticos?",
        "keywords_esperadas": ["energías renovables", "solar fotovoltaica"]
    }
]

pd.DataFrame(preguntas_prueba)

,pregunta,keywords_esperadas
0,¿Qué importancia tiene el sector minero energé...,"[7%, PIB, 34%, IED, 56%, exportaciones]"
1,¿Qué porcentaje de las exportaciones represent...,"[56%, exportaciones]"
2,¿Cuáles son los principales sectores de consum...,"[transporte, industrial, residencial]"
3,¿Qué objetivo tiene el Plan Energético Naciona...,"[modelo energético, sostenible, 2050]"
4,¿Qué fuentes de energía aumentan su participac...,"[energías renovables, solar fotovoltaica]"


## 7. Evaluación del retrieval con Recall@k

La métrica utilizada es Recall@k.  
Esta métrica evalúa si, dentro de los primeros k chunks recuperados, aparece evidencia asociada con la respuesta esperada.

En este proyecto, se usa una evaluación basada en palabras clave esperadas. Esto permite verificar si los chunks recuperados contienen información relevante para responder la pregunta.

In [22]:
def evaluar_recall_at_k_keywords(preguntas_prueba, k=5):
    resultados_eval = []

    for item in preguntas_prueba:
        pregunta = item["pregunta"]
        keywords = [kw.lower() for kw in item["keywords_esperadas"]]

        chunks_recuperados = retrieve_chunks(pregunta, top_k=k)

        textos_recuperados = " ".join([
            chunk["texto_chunk"].lower()
            for chunk in chunks_recuperados
        ])

        keywords_encontradas = [
            kw for kw in keywords
            if kw in textos_recuperados
        ]

        encontrado = len(keywords_encontradas) > 0

        resultados_eval.append({
            "pregunta": pregunta,
            "keywords_esperadas": item["keywords_esperadas"],
            f"recall@{k}": int(encontrado),
            "keywords_encontradas": keywords_encontradas,
            "documentos_recuperados": [
                f'{c["documento"]} - pág. {c["pagina"]}'
                for c in chunks_recuperados
            ]
        })

    return pd.DataFrame(resultados_eval)

In [23]:
df_eval = evaluar_recall_at_k_keywords(preguntas_prueba, k=5)

df_eval

,pregunta,keywords_esperadas,recall@5,keywords_encontradas,documentos_recuperados
0,¿Qué importancia tiene el sector minero energé...,"[7%, PIB, 34%, IED, 56%, exportaciones]",1,"[7%, pib, 34%, ied, 56%, exportaciones]",[Sector_Minero_Energetico.pdf_parte_0 - pág. 4...
1,¿Qué porcentaje de las exportaciones represent...,"[56%, exportaciones]",1,"[56%, exportaciones]",[Sector_Minero_Energetico.pdf_parte_0 - pág. 4...
2,¿Cuáles son los principales sectores de consum...,"[transporte, industrial, residencial]",1,"[transporte, industrial, residencial]","[PEN_2050_UPME.pdf_parte_0 - pág. 13, PEN_2050..."
3,¿Qué objetivo tiene el Plan Energético Naciona...,"[modelo energético, sostenible, 2050]",1,"[modelo energético, sostenible, 2050]","[PEN_2050_UPME.pdf_parte_1 - pág. 21, PEN_2050..."
4,¿Qué fuentes de energía aumentan su participac...,"[energías renovables, solar fotovoltaica]",0,[],"[PEN_2050_UPME.pdf_parte_3 - pág. 56, PEN_2050..."


In [24]:
recall_k = df_eval["recall@5"].mean()

print(f"Recall@5: {recall_k:.2f}")

Recall@5: 0.80


### Metricas adicionales: Precision@k y MRR

- **Precision@k**: de los k chunks recuperados, que fraccion contiene al menos una keyword esperada?
- **MRR (Mean Reciprocal Rank)**: inverso de la posicion del primer chunk relevante. Penaliza cuando el chunk mas pertinente aparece tarde en el ranking.

In [25]:
def precision_at_k(chunks_recuperados, keywords_esperadas, k=5):
    keywords = [kw.lower() for kw in keywords_esperadas]
    relevantes = sum(
        any(kw in chunk["texto_chunk"].lower() for kw in keywords)
        for chunk in chunks_recuperados[:k]
    )
    return relevantes / k

def reciprocal_rank(chunks_recuperados, keywords_esperadas):
    keywords = [kw.lower() for kw in keywords_esperadas]
    for rank, chunk in enumerate(chunks_recuperados, start=1):
        if any(kw in chunk["texto_chunk"].lower() for kw in keywords):
            return 1.0 / rank
    return 0.0

resultados_ext = []
for item in preguntas_prueba:
    chunks = retrieve_chunks(item["pregunta"], top_k=5)
    resultados_ext.append({
        "Pregunta": item["pregunta"][:55] + "...",
        "Recall@5": int(any(
            kw.lower() in " ".join(c["texto_chunk"] for c in chunks).lower()
            for kw in item["keywords_esperadas"]
        )),
        "Precision@5": round(precision_at_k(chunks, item["keywords_esperadas"], k=5), 2),
        "RR@5": round(reciprocal_rank(chunks, item["keywords_esperadas"]), 3),
    })

df_metricas = pd.DataFrame(resultados_ext)
display(df_metricas)

print(f"\nPromedios:")
print(f"  Recall@5:    {df_metricas['Recall@5'].mean():.3f}")
print(f"  Precision@5: {df_metricas['Precision@5'].mean():.3f}")
print(f"  MRR@5:       {df_metricas['RR@5'].mean():.3f}")

,Pregunta,Recall@5,Precision@5,RR@5
0,¿Qué importancia tiene el sector minero energé...,1,0.8,1.0
1,¿Qué porcentaje de las exportaciones represent...,1,0.2,1.0
2,¿Cuáles son los principales sectores de consum...,1,0.8,1.0
3,¿Qué objetivo tiene el Plan Energético Naciona...,1,1.0,1.0
4,¿Qué fuentes de energía aumentan su participac...,0,0.0,0.0



Promedios:
  Recall@5:    0.800
  Precision@5: 0.560
  MRR@5:       0.800


## Análisis del Recall@5

El sistema obtiene un Recall@5 alto, lo que indica que el índice vectorial logra recuperar fragmentos relevantes para las preguntas de prueba.

Sin embargo, esta métrica evalúa únicamente la etapa de recuperación de información. No garantiza que la respuesta generada por el modelo sea siempre perfecta, ya que la generación puede presentar errores de interpretación, especialmente cuando hay muchas cifras cercanas en un mismo fragmento.

## 8. Diagnóstico de retrieval

A continuación se muestran los tres primeros casos de prueba, junto con los chunks recuperados, sus documentos, páginas y scores de similitud.

Esto permite analizar si el sistema está recuperando fragmentos coherentes con cada pregunta.

In [26]:
for i in range(3):
    pregunta = preguntas_prueba[i]["pregunta"]
    resultados = retrieve_chunks(pregunta, top_k=3)

    print("\n==============================")
    print("Pregunta:", pregunta)

    for r in resultados:
        print(f"Doc: {r['documento']} | Página: {r['pagina']} | Score: {r['score']:.2f}")
        print(r["texto_chunk"][:400])
        print("-" * 80)


Pregunta: ¿Qué importancia tiene el sector minero energético para la economía colombiana?
Doc: Sector_Minero_Energetico.pdf_parte_0 | Página: 4 | Score: 0.84
Proviene de los sectores de Minería, Hidrocarburos y Energía Eléctrica de la IED de las exportaciones Minero - energéticos representaron El sector Minero Energético es clave para la reactivación económica de colombia 7% 34% 56% del pib nacional En 2019 los Sectores Proviene del sector de los servicios públicos 5,7% Entre 2018 -2019 la IED en el sector minero energético creció 21% de los ingresos d
--------------------------------------------------------------------------------
Doc: Sector_Minero_Energetico.pdf_parte_1 | Página: 27 | Score: 0.79
La minería genera empleo para muchos colombianos empleos La actividad minera se desarrolla en municipios alejados de los centros urbanos, generando empleos y desarrollo. 350.000 Guajira 38% PIB Carbón térmico Cesar 39% PIB Carbón térmico Chocó 11% PIB Oro y Cobre Córdoba 7%PIB Níquel y Oro

## Análisis del diagnóstico de retrieval

A partir de los tres primeros casos evaluados, se observa que el sistema logra recuperar correctamente fragmentos relevantes del corpus.

Por ejemplo:

- Para la pregunta sobre la importancia del sector minero energético, el sistema recupera el documento correcto donde se mencionan indicadores clave como el 7% del PIB, el 34% de la inversión extranjera directa y el 56% de las exportaciones. :contentReference[oaicite:0]{index=0}  

- Para la pregunta sobre exportaciones, el sistema también recupera correctamente el fragmento donde aparece el 56%, lo cual indica una buena correspondencia semántica entre la pregunta y los embeddings.

- En el caso de los sectores de consumo de energía, aunque los resultados son relevantes, se observa que algunos chunks contienen información más general, lo que puede afectar la precisión de la respuesta final.

En general, el retrieval muestra buen desempeño, pero su precisión depende de qué tan específica sea la pregunta.

## Prueba de 5 preguntas con chunks mostrados

A continuación se prueban las cinco preguntas definidas y se muestran los principales chunks recuperados para cada una. Esto permite verificar manualmente si el sistema está trayendo evidencia útil para responder.

In [27]:
for i, item in enumerate(preguntas_prueba, start=1):
    pregunta = item["pregunta"]
    resultados = retrieve_chunks(pregunta, top_k=3)

    print("=" * 100)
    print(f"PREGUNTA {i}: {pregunta}")
    print("=" * 100)

    for j, r in enumerate(resultados, start=1):
        print(f"\nChunk {j}")
        print(f"Documento: {r['documento_original']} | Segmento: {r['documento']} | Página: {r['pagina']}")
        print(f"Score: {r['score']:.3f}")
        print("Texto recuperado:")
        print(r["texto_chunk"][:800])
        print("-" * 100)

PREGUNTA 1: ¿Qué importancia tiene el sector minero energético para la economía colombiana?

Chunk 1
Documento: Sector_Minero_Energetico.pdf | Segmento: Sector_Minero_Energetico.pdf_parte_0 | Página: 4
Score: 0.842
Texto recuperado:
Proviene de los sectores de Minería, Hidrocarburos y Energía Eléctrica de la IED de las exportaciones Minero - energéticos representaron El sector Minero Energético es clave para la reactivación económica de colombia 7% 34% 56% del pib nacional En 2019 los Sectores Proviene del sector de los servicios públicos 5,7% Entre 2018 -2019 la IED en el sector minero energético creció 21% de los ingresos de la nación 12% @DiegoMesaP
----------------------------------------------------------------------------------------------------

Chunk 2
Documento: Sector_Minero_Energetico.pdf | Segmento: Sector_Minero_Energetico.pdf_parte_1 | Página: 27
Score: 0.789
Texto recuperado:
La minería genera empleo para muchos colombianos empleos La actividad minera se desarrolla en mu

PREGUNTA 5: ¿Qué fuentes de energía aumentan su participación en los escenarios energéticos?

Chunk 1
Documento: PEN_2050_UPME.pdf | Segmento: PEN_2050_UPME.pdf_parte_3 | Página: 56
Score: 0.760
Texto recuperado:
a gas natural para reducir las emisiones del sector. (UPME - UT INCOMBUSTION, 2014). En los subsectores de refinerías, minerales n o metálicos y productos metalúrgicos, la energía térmica necesaria en los procesos es obtenida principalmente de usos de calor directo, es decir de hornos de temperaturas medias y bajas. De acuerdo con el estudio de Balance de Energía Útil realizado por la UPME en 201942, las aplicaciones de calor directo existe un potencial de aumento de eficiencia entre 23,6% y 45,45% y si se considera que los subsectores previamente mencionados, que representan 44% de la demanda de energía térmica de la industria manufac turera, se pueden obtener altas reducciones en emisiones si se hacen sustituciones de hornos que funcionan con 42 UPME-IREES - TEP - CORPOEMA (

## Comentario sobre los chunks recuperados

En las cinco preguntas se observa que el sistema recupera fragmentos relacionados con los temas consultados. Las preguntas más específicas, como las relacionadas con porcentajes de exportaciones o participación del sector minero energético, tienden a recuperar chunks más precisos.

En cambio, preguntas más generales pueden recuperar fragmentos relacionados pero menos directos, lo cual muestra una limitación común en sistemas RAG: la calidad del retrieval depende de la especificidad de la pregunta y de cómo está segmentado el corpus.

## 9. Mapa de calor de similitud coseno

Se construye un mapa de calor para visualizar la similitud entre las preguntas de prueba y una muestra de chunks del corpus.

Esto permite observar qué tan cercanas semánticamente son las preguntas frente a los fragmentos recuperables.

In [28]:
queries = [p["pregunta"] for p in preguntas_prueba]

query_embeddings = embedding_model.encode(queries, convert_to_numpy=True)
faiss.normalize_L2(query_embeddings)

# Se toma una muestra de los primeros 50 chunks para que el gráfico sea legible
sample_embeddings = embeddings[:50]

similarity_matrix = np.dot(query_embeddings, sample_embeddings.T)

plt.figure(figsize=(14, 6))
sns.heatmap(similarity_matrix, cmap="viridis")

plt.title("Mapa de calor de similitud coseno entre preguntas y chunks")
plt.xlabel("Chunks")
plt.ylabel("Preguntas")
plt.show()

C:\Users\sebas\AppData\Local\Temp\ipykernel_8456\2192921495.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Análisis del mapa de calor de similitud

El mapa de calor muestra la similitud coseno entre las preguntas de prueba y los chunks del corpus.

Se pueden observar zonas con mayor intensidad (colores más claros), lo que indica una mayor similitud semántica entre ciertas preguntas y fragmentos del texto.

Esto evidencia que:

- El modelo de embeddings logra capturar relaciones semánticas entre preguntas y documentos.
- Existen múltiples chunks relevantes para una misma pregunta, lo cual es positivo para el recall.
- Algunas preguntas presentan similitud distribuida en varios fragmentos, lo que indica que la información relevante puede estar dispersa en el corpus.

Este comportamiento es esperado en sistemas RAG y refuerza la necesidad de recuperar múltiples chunks (top-k) en lugar de uno solo.

## 10. Generación de respuestas (RAG)

En esta etapa se utiliza un modelo generativo para construir respuestas a partir de los chunks recuperados.

El modelo recibe como entrada:
- La pregunta del usuario
- El contexto compuesto por los fragmentos recuperados

El objetivo es generar una respuesta coherente basada únicamente en la información disponible en el contexto.

In [29]:
def build_context(chunks):
    context = ""
    
    for i, chunk in enumerate(chunks, start=1):
        context += f"[Fuente {i} | {chunk['documento_original']} | Página {chunk['pagina']}]\n"
        context += chunk["texto_chunk"] + "\n\n"
    
    return context

In [30]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=0 if DEVICE == "cuda" else -1,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
# Sobrescribir generation_config del modelo para evitar conflictos
generator.model.generation_config.max_new_tokens = 180
generator.model.generation_config.do_sample = False
generator.model.generation_config.max_length = None
print(f"Modelo generativo cargado en: {DEVICE.upper()}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  43%|████▎     | 124/290 [00:00<00:00, 1233.49it/s]

Loading weights:  86%|████████▌ | 248/290 [00:00<00:00, 972.46it/s] 

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 977.29it/s]

Modelo generativo cargado en: CUDA


In [31]:
def rag_answer(query, top_k=3):
    chunks = retrieve_chunks(query, top_k=top_k)
    context = build_context(chunks)

    prompt = f"""
Responde en espanol usando unicamente el contexto.
Si la respuesta no esta en el contexto, di: No se encuentra informacion suficiente.

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""

    output = generator(prompt)[0]["generated_text"]
    respuesta = output.split("Respuesta:")[-1].strip()

    return {
        "pregunta": query,
        "respuesta": respuesta,
        "fuentes": [
            f'{c["documento_original"]} - pagina {c["pagina"]}'
            for c in chunks
        ],
    }

## Ejemplo de respuesta


In [32]:
resultado = rag_answer(
    "¿Qué importancia tiene el sector minero energético para la economía colombiana?",
    top_k=3
)

print("Pregunta:", resultado["pregunta"])
print("Respuesta:", resultado["respuesta"])
print("Fuentes:", resultado["fuentes"])

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pregunta: ¿Qué importancia tiene el sector minero energético para la economía colombiana?
Respuesta: El sector minero energético representa un importante componente económico para Colombia. Según el informe, este sector contribuye significativamente al PIB nacional, con una contribución de aproximadamente 7% al PIB nacional en 2019. Además, el sector minero energético también genera empleo y contribuye significativamente al desarrollo económico, con una participación de entre 350.000 y 80.000 empleos en la región. También menciona que el sector minero energético es importante para el comercio exterior debido a su importancia en el mercado internacional de carbón y otros productos de energía. Sin embargo, hay algunas limitaciones, como el alto costo de la minería y la dependencia de los recursos naturales, lo cual puede tener consecuencias negativas para el bienestar social y ambiental de la población. Por lo tanto, es crucial que se realicen inversiones en tecnologías renovables y en i

## 11. Análisis de errores

Se observa que, aunque el sistema recupera correctamente los fragmentos relevantes (alto Recall@5), el modelo generativo puede presentar errores en la interpretación del contexto.

Ejemplo observado:

- El modelo mezcla valores numéricos, confundiendo el 7% del PIB con el 56% de exportaciones.

Esto ocurre porque:

- Los chunks contienen múltiples cifras en el mismo fragmento.
- El modelo generativo no siempre distingue claramente la relación entre valores y conceptos.
- No existe un mecanismo de validación posterior de la respuesta.

Esto evidencia que el rendimiento del sistema RAG no depende únicamente del retrieval, sino también de la capacidad del modelo generativo para interpretar correctamente el contexto.

## 12. Limitaciones del sistema

El sistema presenta las siguientes limitaciones:

- El modelo generativo puede mezclar información cuando hay múltiples cifras en el contexto.
- El chunking basado en palabras puede separar información relevante o agrupar demasiados conceptos en un mismo fragmento.
- No se realiza validación estructurada de las respuestas generadas.
- El corpus es limitado, lo que puede afectar la cobertura de las respuestas.

## 13. Mejoras futuras

Se proponen las siguientes mejoras, ordenadas por impacto esperado:

### Tecnicas (implementables en el notebook)

- **Chunking semantico**: usar limites de parrafo o deteccion de secciones en lugar de ventana de palabras fija.
- **Persistencia del indice**: guardar y cargar el indice FAISS desde disco para evitar recomputacion (implementado en este notebook).
- **Ampliar corpus**: incorporar mas documentos del sector energetico colombiano.
- **Validacion de respuestas**: verificar si los valores numericos citados coinciden con los del contexto recuperado.

### Modelos alternativos

Para mejorar la calidad de generacion, alternativas al modelo Qwen2.5-0.5B:

| Modelo | Tamano | Ventaja |
|---|---|---|
| `meta-llama/Llama-3.2-3B-Instruct` | 3B | Mejor razonamiento, aun liviano |
| `google/gemma-2-2b-it` | 2B | Buen rendimiento multilingue, Apache 2.0 |
| `groq/llama3-8b-8192` (API) | 8B via API | Gratuito, latencia muy baja (Groq Cloud) |
| `claude-haiku-4-5` (API) | — | Alta precision en extraccion de datos numericos |

Para los embeddings, alternativas al modelo actual `paraphrase-multilingual-MiniLM-L12-v2`:

| Modelo | Dimensiones | Ventaja |
|---|---|---|
| `intfloat/multilingual-e5-large` | 1024 | Mayor precision semantica |
| `BAAI/bge-m3` | 1024 | State-of-the-art multilingue |

### Frameworks especializados

Para una implementacion mas robusta o en produccion:

- **LangChain** (`langchain.com`): abstraccion completa de pipelines RAG. Incluye loaders para multiples formatos, text splitters semanticos, integracion con decenas de vector stores y LLMs.

  ```python
  from langchain.document_loaders import PyPDFLoader
  from langchain.text_splitter import RecursiveCharacterTextSplitter
  from langchain_community.vectorstores import FAISS
  from langchain.chains import RetrievalQA
  ```

- **LlamaIndex** (`llamaindex.ai`): especializado en RAG sobre documentos. Soporta indices jerarquicos, query engines avanzados y evaluacion nativa de faithfulness y relevancia.

- **Haystack** (`haystack.deepset.ai`): pipeline RAG orientado a produccion con retrieval hibrido (denso + disperso) y evaluacion con datasets de referencia.

La implementacion desde cero en este notebook es valida para el contexto academico, pues demuestra el funcionamiento interno del pipeline sin abstracciones intermedias.

## 14. Conclusión

El sistema RAG desarrollado logra recuperar de manera efectiva información relevante del corpus, alcanzando un alto Recall@5.

Esto demuestra que el uso de embeddings junto con FAISS es una estrategia adecuada para tareas de búsqueda semántica en documentos técnicos.

Sin embargo, se identificaron limitaciones en la etapa de generación, donde el modelo puede interpretar incorrectamente ciertos elementos del contexto.

En general, el enfoque RAG permite mejorar el acceso a información en documentos complejos, pero requiere ajustes adicionales para garantizar la precisión de las respuestas generadas.

## Limpieza de memoria

Se libera la memoria del modelo generativo al finalizar el notebook.

In [33]:
# Liberar memoria del modelo generativo
del generator
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("Cache GPU liberada")
print("Memoria liberada correctamente")

Cache GPU liberada
Memoria liberada correctamente
